In [2]:
import numpy as np
import pandas as pd
import nltk
import string

In [4]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Abhi0072/HateXplain")

Repo card metadata block was not found. Setting CardData to empty.


In [5]:
# Access the 'train' split
train_dataset = ds['train']

# Convert to pandas DataFrame
df = train_dataset.to_pandas()

df.sample(20)



,text,label
4602,do some research tony you will find out that m...,hatespeech
1751,no <number> million white children raped by pa...,offensive
8473,you all taking harassment to another level out...,normal
12182,<user> <user> arabs are so fukin idiots,hatespeech
9940,you on the blade and you still broke what a si...,offensive
6784,keenly creator confront warlock dyke,normal
5028,<user> <user> hell no except if he puts everyt...,normal
7644,yeah i already knew oprah was a cunt a stink n...,hatespeech
7625,<date> is a tuesday the hefty pric ei must pay...,normal
9117,<user> if you were the police officer getting ...,normal


# Basic Preprocessing

In [6]:
import nltk
import string
import re

In [7]:
from nltk.corpus import stopwords
stop = set(stopwords.words('english'))


In [8]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [9]:
def transform(text):
    text = re.sub(r'<user>', '', text)
    text = text.lower()
    tokens = nltk.word_tokenize(text)

    tokens = [word for word in tokens if word.isalnum()]
    tokens = [word for word in tokens if word not in stop]
    tokens = [ps.stem(word) for word in tokens]

    return " ".join(tokens)

In [10]:
text = "I loved the lectures on yt they were really great"
transform(text)

'love lectur yt realli great'

In [11]:
df['trans_text'] = df['text'].apply(transform)
df.head()

,text,label,trans_text
0,i dont think im getting my baby them white 9 h...,normal,dont think im get babi white 9 two white j nik...
1,we cannot continue calling ourselves feminists...,normal,continu call feminist right womxn arent addres...
2,nawt yall niggers ignoring me,normal,nawt yall nigger ignor
3,<user> i am bit confused coz chinese ppl can n...,hatespeech,bit confus coz chines ppl access twitter thn c...
4,this bitch in whataburger eating a burger with...,hatespeech,bitch whataburg eat burger top bun hold palm h...


In [12]:
from sklearn.preprocessing import LabelEncoder
ln = LabelEncoder()
df['label'] = ln.fit_transform(df['label'])

In [31]:
df['label']

0        1
1        1
2        1
3        0
4        0
        ..
20143    2
20144    2
20145    2
20146    0
20147    0
Name: label, Length: 20148, dtype: int64

# Text Preprocessing

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

In [14]:
df.shape

(20148, 3)

In [15]:
# cv = CountVectorizer()
tfi = TfidfVectorizer(max_features=2000)

In [16]:
Xtfi = tfi.fit_transform(df['trans_text']).toarray()
Xtfi.shape

(20148, 2000)

In [17]:
y = df['label']

In [35]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(Xtfi,y,random_state=42,test_size=0.4)
X_train.shape

(12088, 2000)

In [36]:
from sklearn.metrics import accuracy_score,precision_score,confusion_matrix

In [37]:
from xgboost import XGBClassifier
xgb_clf = XGBClassifier(
    objective='multi:softmax',   
    num_class=3,                 
    eval_metric='mlogloss',      
    use_label_encoder=False,     
    n_estimators=150,            
    max_depth=6,                 
    learning_rate=0.1,
    random_state=123
)
xgb_clf.fit(X_train, y_train)
y_pred = xgb_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("precision (macro):", precision_score(y_test, y_pred, average='macro'))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

c:\SentiChat\model_env\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:00:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.6053349875930522
precision (macro): 0.6078916666260538

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.59      0.65      2399
           1       0.59      0.81      0.68      3312
           2       0.51      0.34      0.41      2349

    accuracy                           0.61      8060
   macro avg       0.61      0.58      0.58      8060
weighted avg       0.61      0.61      0.59      8060


Confusion Matrix:
 [[1404  673  322]
 [ 196 2669  447]
 [ 336 1207  806]]


In [43]:
label_map = {
    0: "hatespeech",
    2: "offensive",
    1: "normal"
}

def predict_sentiment(text, model, tfidf, label_map, stop_words):

    processed = transform(text)

    vector = tfidf.transform([processed])

    pred = model.predict(vector)[0]

    return label_map[pred]


predict_sentiment("i am a good boy",xgb_clf,tfi,label_map,stop)


'offensive'

In [ ]:
import pickle

def save_artifacts(model, tfidf, label_map):

    with open("model.pkl", "wb") as f:
        pickle.dump(xgb_clf, f)

    with open("tfidf.pkl", "wb") as f:
        pickle.dump(tfi, f)

    with open("label_map.pkl", "wb") as f:
        pickle.dump(label_map, f)

    print("Artifacts saved successfully.")